In [ ]:
import os, sys, time
from pathlib import Path
ROOT = Path(os.environ.get("REPO_ROOT", "/kaggle/working/ML-for-Wireless-Security-Threats-Identification"))
sys.path.insert(0, str(ROOT))
RESULTS_DIRECTORY, SAVED_MODEL_DIRECTORY = Path("results"), Path("saved_model")
RESULTS_DIRECTORY.mkdir(exist_ok=True)
SAVED_MODEL_DIRECTORY.mkdir(exist_ok=True)
import numpy as np
from shared.config import LABELS, SEED
from shared.data_loader import load_splits
from shared.evaluator import measure_latency_milliseconds, save_json, save_results
from shared.tracer import build_alerts, save_bundle
import torch
from torch import nn
from shared.data_loader import fit_standardizer, standardize_in_place
from shared.evaluator import plot_training_curve
from shared.tracer import make_mlp
device = "cuda" if torch.cuda.is_available() else "cpu"
HIDDEN_SIZES, EPOCHS, BATCH_SIZE = (256, 128), 15, 2048
torch.manual_seed(SEED)

In [ ]:
import psutil
def memory_gb():
    return round(psutil.Process().memory_info().rss / 1e9, 1)

splits = load_splits(as_matrix=True, train_cap_per_label=1_000_000)
print("sau load_splits:", memory_gb(), "GB", splits["features_train"].shape)
mean, scale = fit_standardizer(splits["features_train"])
for name in ("features_train", "features_val", "features_test"):
    standardize_in_place(splits[name], mean, scale)
print("sau standardize:", memory_gb(), "GB")
tensor_train = torch.from_numpy(splits["features_train"])
tensor_val = torch.from_numpy(splits["features_val"])
tensor_test = torch.from_numpy(splits["features_test"])
target_train = torch.tensor(splits["labels_train"].to_numpy())
target_val = splits["labels_val"].to_numpy()
class_counts = np.bincount(splits["labels_train"], minlength=len(LABELS)).astype("float32")
class_weights = torch.tensor(class_counts.sum() / np.maximum(class_counts, 1) / len(LABELS), device=device)

In [ ]:
def predict_probabilities(network, tensor):
    with torch.no_grad():
        chunks = [torch.softmax(network(tensor[start:start + 8192].to(device)), 1).cpu()
                  for start in range(0, len(tensor), 8192)]
    return torch.cat(chunks).numpy()

network = make_mlp(tensor_train.shape[1], len(LABELS), HIDDEN_SIZES).to(device)
optimizer = torch.optim.Adam(network.parameters(), lr=1e-3)
loss_function = nn.CrossEntropyLoss(weight=class_weights)
history = {"loss": [], "val_accuracy": []}
start_time = time.perf_counter()
for epoch in range(EPOCHS):
    network.train()
    permutation = torch.randperm(len(tensor_train))
    total_loss = 0.0
    for start in range(0, len(permutation), BATCH_SIZE):
        batch_indices = permutation[start:start + BATCH_SIZE]
        optimizer.zero_grad()
        loss = loss_function(network(tensor_train[batch_indices].to(device)), target_train[batch_indices].to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch_indices)
    network.eval()
    history["loss"].append(total_loss / len(permutation))
    history["val_accuracy"].append(float((predict_probabilities(network, tensor_val).argmax(1) == target_val).mean()))
    print(epoch, history["loss"][-1], history["val_accuracy"][-1])
train_time = time.perf_counter() - start_time

In [ ]:
probabilities = predict_probabilities(network, tensor_test)
latency = measure_latency_milliseconds(lambda tensor: predict_probabilities(network, tensor), tensor_test)
metrics = save_results(RESULTS_DIRECTORY, "mlp", splits["labels_test"], probabilities.argmax(1), train_time, latency)
plot_training_curve(history, RESULTS_DIRECTORY / "training_curve.png")
torch.save({
    "state_dict": {name: value.cpu() for name, value in network.state_dict().items()},
    "schema": splits["schema"], "mean": mean.tolist(), "scale": scale.tolist(),
    "input_size": tensor_train.shape[1], "output_size": len(LABELS), "hidden_sizes": list(HIDDEN_SIZES),
}, SAVED_MODEL_DIRECTORY / "model.pt")

In [ ]:
alerts = build_alerts(splits["metadata_test"], probabilities.argmax(axis=1), probabilities.max(axis=1))
save_json(alerts, RESULTS_DIRECTORY / "trace_results.json")
print(metrics)
print(len(alerts), "incidents")